# Prompt Engineering Best Practices: Chain of Thought Reasoning

Maintained runnable companion. Set OPENAI_API_KEY privately in the environment or an uncommitted .env file; never put keys in cells. All model calls incur charges. Originals are retained in the nonexecuting appendix. The notebook is output-free in Git.

In [ ]:
import os, json, re, sys
from importlib.metadata import version
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=60)
def get_completion_from_messages(messages, model=MODEL, temperature=None):
    options = {} if temperature is None else {"temperature": temperature}
    result = client.chat.completions.create(
        model=model, messages=messages, reasoning_effort="none",
        max_completion_tokens=2048, **options)
    if result.choices[0].finish_reason != "stop":
        raise RuntimeError("The completion did not finish normally")
    content = result.choices[0].message.content
    if not content:
        raise RuntimeError("No text returned")
    return content
def get_completion(prompt, model=MODEL, temperature=None):
    return get_completion_from_messages(
        [{"role": "user", "content": prompt}], model=model, temperature=temperature)
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "python-dotenv:", version("python-dotenv"))
print("Model:", MODEL)

In [ ]:
delimiter = "####"

In [ ]:
instruction_intro = 'Return concise factual findings for four application checks, followed by a customer answer. Do not provide a private internal monologue. The customer query is delimited by ####.'

In [ ]:
step_one = 'Step 1:#### State briefly whether the query concerns named products or a product category.'

In [ ]:
catalog = 'All available products: \n1. Product: TechPro Ultrabook\n   Category: Computers and Laptops\n   Brand: TechPro\n   Model Number: TP-UB100\n   Warranty: 1 year\n   Rating: 4.5\n   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor\n   Description: A sleek and lightweight ultrabook for everyday use.\n   Price: $799.99\n\n2. Product: BlueWave Gaming Laptop\n   Category: Computers and Laptops\n   Brand: BlueWave\n   Model Number: BW-GL200\n   Warranty: 2 years\n   Rating: 4.7\n   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060\n   Description: A high-performance gaming laptop for an immersive experience.\n   Price: $1199.99\n\n3. Product: PowerLite Convertible\n   Category: Computers and Laptops\n   Brand: PowerLite\n   Model Number: PL-CV300\n   Warranty: 1 year\n   Rating: 4.3\n   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge\n   Description: A versatile convertible laptop with a responsive touchscreen.\n   Price: $699.99\n\n4. Product: TechPro Desktop\n   Category: Computers and Laptops\n   Brand: TechPro\n   Model Number: TP-DT500\n   Warranty: 1 year\n   Rating: 4.4\n   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660\n   Description: A powerful desktop computer for work and play.\n   Price: $999.99\n\n5. Product: BlueWave Chromebook\n   Category: Computers and Laptops\n   Brand: BlueWave\n   Model Number: BW-CB100\n   Warranty: 1 year\n   Rating: 4.1\n   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS\n   Description: A compact and affordable Chromebook for everyday tasks.\n   Price: $249.99'
step_two = "Step 2:#### Match named products to the catalog and cite relevant listed facts."

In [ ]:
step_three = 'Step 3:#### Briefly identify any customer assumption relevant to the catalog facts.'

In [ ]:
step_four = 'Step 4:#### Briefly check that assumption against listed facts; include any needed price difference.'

In [ ]:
answer_instruction = 'Politely correct incorrect assumptions, using only the five catalog products. For category questions, answer directly from the catalog even if no specific product is named. Always include a friendly final customer answer.'

In [ ]:
output_format = 'Use exactly this format, with each heading on its own line and one brief factual finding per step:\nStep 1:#### <query type>\nStep 2:#### <catalog match>\nStep 3:#### <relevant assumption or none>\nStep 4:#### <catalog check>\nResponse to user:#### <customer answer>\nDo not use #### elsewhere. Always include all headings, including for category questions.'

In [ ]:
system_message = "\n\n".join([instruction_intro, step_one, step_two, catalog, step_three, step_four, answer_instruction, output_format])
print("Catalog products:", catalog.count("Product:"))

In [ ]:
user_message = f"""
by how much is the BlueWave Chromebook more expensive  than the TechPro Desktop"""

messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 

response = get_completion_from_messages(messages)
print(response)
price_response = response

In [ ]:
user_message = f"""
do you sell tvs"""
messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 
response = get_completion_from_messages(messages)
print(response)
tv_response = response

In [ ]:
FALLBACK = "Sorry, I'm having trouble right now, please try asking another question."
def customer_response(text):
    labels = [f"Step {i}:####" for i in range(1, 5)] + ["Response to user:####"]
    if not isinstance(text, str) or text.count("####") != len(labels):
        return FALLBACK
    pattern = r"\A" + r"\s*".join(re.escape(label) + r"(.*?)" for label in labels) + r"\s*\Z"
    match = re.fullmatch(pattern, text.strip(), flags=re.S)
    if not match or any(not section.strip() for section in match.groups()):
        return FALLBACK
    final = match.groups()[-1].strip()
    if re.search(r"Step [1-4]:|Response to user:", final):
        return FALLBACK
    return final
print(customer_response(tv_response))

In [ ]:
from decimal import Decimal
assert catalog.count("Product:") == 5
assert Decimal("999.99") - Decimal("249.99") == Decimal("750.00")
price_final = customer_response(price_response)
tv_final = customer_response(tv_response)
assert price_final != FALLBACK and tv_final != FALLBACK
assert "750" in price_final and any(w in price_final.lower() for w in ["less", "cheaper"])
assert "chromebook" in price_final.lower() and "desktop" in price_final.lower()
tv_normal = tv_final.lower().replace("’", "'")
assert "tv" in tv_normal and any(w in tv_normal for w in ["not", "don't", "do not"])
valid = "\n".join([f"Step {i}:#### factual finding" for i in range(1,5)] + ["Response to user:#### safe answer"])
assert customer_response(valid) == "safe answer"
for bad in [None, "", "private intermediate text", "Step 1:#### private intermediate text", valid.replace("Response to user:#### safe answer", ""), valid.replace("safe answer", ""), valid + "\nResponse to user:#### duplicate", valid.replace("safe answer", "Step 1: hidden text"), valid.replace("factual finding", "", 1), valid.replace("Step 2", "Step 3", 1)]:
    assert customer_response(bad) == FALLBACK
print("PASS: five complete fictional products, both actual queries, independent $750 difference, all required sections and fail-closed missing/empty/duplicate/misordered-marker tests. Visible findings are application output, not private model reasoning.")

## Original source appendix (not executed)

Historical code and transcripts are retained verbatim below, not instructions to run obsolete APIs or paste credentials.

### Original pre block 0

````text
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key  = os.getenv('OPENAI_API_KEY')
````

### Original pre block 1

````text
def get_completion_from_messages(messages, 
                                 model="gpt-3.5-turbo", 
                                 temperature=0, max_tokens=500):

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
        max_tokens=max_tokens, 
    )
    return response.choices[0].message.content
````

### Original pre block 2

````text
delimiter = "####"
````

### Original pre block 3

````text
system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,\
i.e. {delimiter}.
````

### Original pre block 4

````text
Step 1:{delimiter} First decide whether the user is \
asking a question about a specific product or products. \
Product cateogry doesn't count.
````

### Original pre block 5

````text
Step 2:{delimiter} If the user is asking about \
specific products, identify whether \
the products are in the following list.
All available products: 
1. Product: TechPro Ultrabook
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-UB100
   Warranty: 1 year
   Rating: 4.5
   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor
   Description: A sleek and lightweight ultrabook for everyday use.
   Price: $799.99

2. Product: BlueWave Gaming Laptop
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-GL200
   Warranty: 2 years
   Rating: 4.7
   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060
   Description: A high-performance gaming laptop for an immersive experience.
   Price: $1199.99

3. Product: PowerLite Convertible
   Category: Computers and Laptops
   Brand: PowerLite
   Model Number: PL-CV300
   Warranty: 1 year
   Rating: 4.3
   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge
   Description: A versatile convertible laptop with a responsive touchscreen.
   Price: $699.99

4. Product: TechPro Desktop
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-DT500
   Warranty: 1 year
   Rating: 4.4
   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660
   Description: A powerful desktop computer for work and play.
   Price: $999.99

5. Product: BlueWave Chromebook
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-CB100
   Warranty: 1 year
   Rating: 4.1
   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS
   Description: A compact and affordable Chromebook for everyday tasks.
   Price: $249.99
````

### Original pre block 6

````text
Step 3:{delimiter} If the message contains products \
in the list above, list any assumptions that the \
user is making in their \
message e.g. that Laptop X is bigger than \
Laptop Y, or that Laptop Z has a 2 year warranty.
````

### Original pre block 7

````text
Step 4:{delimiter}: If the user made any assumptions, \
figure out whether the assumption is true based on your \
product information.
````

### Original pre block 8

````text
Step 5:{delimiter}: First, politely correct the \
customer's incorrect assumptions if applicable. \
Only mention or reference products in the list of \
5 available products, as these are the only 5 \
products that the store sells. \
Answer the customer in a friendly tone.
````

### Original pre block 9

````text
Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
````

### Original pre block 10

````text
delimiter = "####"
system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,\
i.e. {delimiter}. 

Step 1:{delimiter} First decide whether the user is \
asking a question about a specific product or products. \
Product cateogry doesn't count. 

Step 2:{delimiter} If the user is asking about \
specific products, identify whether \
the products are in the following list.
All available products: 
1. Product: TechPro Ultrabook
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-UB100
   Warranty: 1 year
   Rating: 4.5
   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor
   Description: A sleek and lightweight ultrabook for everyday use.
   Price: $799.99

2. Product: BlueWave Gaming Laptop
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-GL200
   Warranty: 2 years
   Rating: 4.7
   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060
   Description: A high-performance gaming laptop for an immersive experience.
   Price: $1199.99

3. Product: PowerLite Convertible
   Category: Computers and Laptops
   Brand: PowerLite
   Model Number: PL-CV300
   Warranty: 1 year
   Rating: 4.3
   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge
   Description: A versatile convertible laptop with a responsive touchscreen.
   Price: $699.99

4. Product: TechPro Desktop
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-DT500
   Warranty: 1 year
   Rating: 4.4
   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660
   Description: A powerful desktop computer for work and play.
   Price: $999.99

5. Product: BlueWave Chromebook
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-CB100
   Warranty: 1 year
   Rating: 4.1
   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS
   Description: A compact and affordable Chromebook for everyday tasks.
   Price: $249.99

Step 3:{delimiter} If the message contains products \
in the list above, list any assumptions that the \
user is making in their \
message e.g. that Laptop X is bigger than \
Laptop Y, or that Laptop Z has a 2 year warranty.

Step 4:{delimiter}: If the user made any assumptions, \
figure out whether the assumption is true based on your \
product information. 

Step 5:{delimiter}: First, politely correct the \
customer's incorrect assumptions if applicable. \
Only mention or reference products in the list of \
5 available products, as these are the only 5 \
products that the store sells. \
Answer the customer in a friendly tone.

Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
"""
````

### Original pre block 11

````text
user_message = f"""
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop"""

messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 

response = get_completion_from_messages(messages)
print(response)
````

### Original pre block 12

````text
user_message = f"""
do you sell tvs"""
messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 
response = get_completion_from_messages(messages)
print(response)
````

### Original pre block 13

````text
try:
    final_response = response.split(delimiter)[-1].strip()
except Exception as e:
    final_response = "Sorry, I'm having trouble right now, please try asking another question."
    
print(final_response)
````

### Original transcript 0

````text
user_message = f”””
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop”””
````

### Original transcript 1

````text
messages = [ 
{‘role’:’system’, 
 ‘content’: system_message}, 
{‘role’:’user’, 
 ‘content’: f”{delimiter}{user_message}{delimiter}”}, 
]
````

### Original transcript 2

````text
response = get_completion_from_messages(messages)
print(response)
user_message = f”””
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop”””
​
messages = [ 
{‘role’:’system’, 
 ‘content’: system_message}, 
{‘role’:’user’, 
 ‘content’: f”{delimiter}{user_message}{delimiter}”}, 
] 
​
response = get_completion_from_messages(messages)
print(response)
Step 1:#### The user is comparing the prices of two specific products.
Step 2:#### The products being compared are:
- BlueWave Chromebook: Price $249.99
- TechPro Desktop: Price $999.99
Step 3:#### The user assumes that the BlueWave Chromebook is more expensive than the TechPro Desktop.
Step 4:#### The assumption made by the user is incorrect. The TechPro Desktop is actually more expensive than the BlueWave Chromebook.
Response to user:#### The TechPro Desktop is actually more expensive than the BlueWave Chromebook. The TechPro Desktop is priced at $999.99, while the BlueWave Chromebook is priced at $249.99.
````

### Original transcript 3

````text
Step 1:#### The user is asking about a specific product category (TVs) and not a specific product. Therefore, the question is not about a specific product.
````

### Original transcript 4

````text
The user is asking about a specific product category (TVs) and not a specific product. Therefore, the question is not about a specific product.
````